# 03 — A Simple Chatbot With Memory (From Scratch)

Companion notebook to `03-chatbot-architecture-azure-openai.md`.

This notebook builds a minimal conversational loop **in pure Python, with no external API and no
LangChain dependency**, to make one point extremely concrete: an LLM has no memory of its own
(Chapter 1) -- your application is entirely responsible for tracking conversation history and
deciding what to do when it outgrows the model's context window.

We simulate the LLM with a simple echo-style function so this runs offline; swap it for a real
`AzureChatOpenAI` call at the end and the memory-management logic doesn't change at all.

## Step 1: A minimal message store

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Message:
    role: str      # "system" | "user" | "assistant"
    content: str

    def token_estimate(self) -> int:
        # crude but serviceable stand-in for a real tokenizer (e.g. tiktoken):
        # ~4 characters per token is a commonly used rule of thumb for English text
        return max(1, len(self.content) // 4)

@dataclass
class ConversationMemory:
    system_prompt: str
    messages: List[Message] = field(default_factory=list)

    def add(self, role: str, content: str) -> None:
        self.messages.append(Message(role, content))

    def total_tokens(self) -> int:
        system_tokens = Message("system", self.system_prompt).token_estimate()
        return system_tokens + sum(m.token_estimate() for m in self.messages)

    def as_prompt_messages(self) -> List[Message]:
        return [Message("system", self.system_prompt)] + self.messages

## Step 2: Why this matters -- the model sees nothing you don't send it

Let's simulate a mock LLM that can *only* answer questions about things explicitly present in the
messages it receives -- deliberately naive, to make the "no hidden memory" point undeniable.

In [ ]:
def mock_llm_call(prompt_messages: List[Message]) -> str:
    """Extremely naive mock: looks for a name mentioned earlier in the *visible*
    message history to answer 'what is my name' style questions. If the name
    was truncated out of history, the mock (correctly) can't find it -- exactly
    like a real LLM that never saw that text in its context window.
    """
    last_user_msg = prompt_messages[-1].content.lower()
    if "my name" in last_user_msg:
        for m in prompt_messages:
            if m.role == "user" and "my name is" in m.content.lower():
                name = m.content.lower().split("my name is")[-1].strip().split()[0]
                return f"Your name is {name.capitalize()}."
        return "I don't have that information in our conversation so far."
    return f"(mock reply to: '{prompt_messages[-1].content}')"

In [ ]:
memory = ConversationMemory(system_prompt="You are a helpful internal assistant.")

memory.add("user", "Hi, my name is Abhishek.")
memory.add("assistant", mock_llm_call(memory.as_prompt_messages()))
print(memory.messages[-1].content)

memory.add("user", "What is my name?")
reply = mock_llm_call(memory.as_prompt_messages())
memory.add("assistant", reply)
print(reply)

## Step 3: Context window truncation

Now let's simulate a long conversation that eventually exceeds a (deliberately tiny, for
demonstration) token budget, and watch what happens to the "what is my name" answer once the
message that introduced the name gets truncated out of the window -- this is exactly the tradeoff
discussed in `03-chatbot-architecture-azure-openai.md` under conversation state management.

In [ ]:
def truncate_to_budget(memory: ConversationMemory, max_tokens: int) -> List[Message]:
    """Sliding-window truncation: always keep the system prompt, then keep as many
    of the *most recent* messages as fit in the remaining token budget. This is the
    simplest real-world strategy; the alternative (mentioned in Chapter 3) is
    periodic summarization of older turns instead of dropping them outright.
    """
    system_msg = Message("system", memory.system_prompt)
    budget = max_tokens - system_msg.token_estimate()

    kept: List[Message] = []
    for m in reversed(memory.messages):  # walk backwards from the most recent message
        cost = m.token_estimate()
        if cost > budget:
            break
        kept.append(m)
        budget -= cost
    kept.reverse()
    return [system_msg] + kept

In [ ]:
long_memory = ConversationMemory(system_prompt="You are a helpful internal assistant.")
long_memory.add("user", "Hi, my name is Abhishek.")
long_memory.add("assistant", "Nice to meet you, Abhishek!")

# pad the conversation with a bunch of unrelated turns to force truncation
filler_topics = [
    "What are your branch hours?", "How do refunds work?", "Where is the nearest branch?",
    "Can I open a joint account online?", "What documents do I need for a loan?",
    "How do I dispute a transaction?", "What is the daily withdrawal limit?",
]
for topic in filler_topics:
    long_memory.add("user", topic)
    long_memory.add("assistant", f"Here is some fairly long boilerplate policy text answering: {topic} " * 3)

long_memory.add("user", "What is my name?")

print(f"Full history token estimate: {long_memory.total_tokens()}")

# a tiny budget, deliberately, to force truncation for this demo
TINY_BUDGET = 120
visible = truncate_to_budget(long_memory, TINY_BUDGET)
print(f"Messages visible to the model after truncation: {len(visible)} of {len(long_memory.messages) + 1}")

reply = mock_llm_call(visible)
print("Model reply:", reply)

With a small enough budget, the message that introduced "my name is Abhishek" gets pushed out of
the visible window, and the mock model (correctly, given what it can see) says it doesn't know the
answer -- a real LLM would behave identically, because it truly cannot see anything outside the
prompt it was given. This is exactly why production chatbots need a deliberate memory strategy
(sliding window vs. summarization) rather than naively growing the history forever.

## Step 4: Swapping in a real LLM

The memory-management code above (`ConversationMemory`, `truncate_to_budget`) is provider-agnostic
-- only `mock_llm_call` needs to change:

```python
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = AzureChatOpenAI(azure_deployment="<your-deployment-name>", temperature=0.2)

def real_llm_call(prompt_messages):
    role_map = {"system": SystemMessage, "user": HumanMessage, "assistant": AIMessage}
    lc_messages = [role_map[m.role](content=m.content) for m in prompt_messages]
    return llm.invoke(lc_messages).content

reply = real_llm_call(truncate_to_budget(long_memory, max_tokens=4000))
```

For production use, replace the `len(content) // 4` token estimate with a real tokenizer (e.g.
`tiktoken`) and consider `RunnableWithMessageHistory` from `langchain_core` once you want this
wired into an LCEL chain -- see `02-langchain-and-lcel.md`.